In [2]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from utiles import read_frames

df = pd.read_csv("index.csv")
device = "cuda" if torch.cuda.is_available() else "cpu"

weights = R2Plus1D_18_Weights.KINETICS400_V1
model = r2plus1d_18(weights=weights)
model.fc = nn.Identity()          # вместо 400 классов Kinetics -> вектор 512
model.eval().to(device)
tf = weights.transforms()         # resize, crop 112x112, нормализация Kinetics
print(tf)

VideoClassification(
    crop_size=[112, 112]
    resize_size=[128, 171]
    mean=[0.43216, 0.394666, 0.37645]
    std=[0.22803, 0.22145, 0.216989]
    interpolation=InterpolationMode.BILINEAR
)


In [3]:
feats = []
with torch.no_grad():
    for path in df.path:
        frames = torch.from_numpy(read_frames(path)).permute(0, 3, 1, 2)  # (16, 3, H, W)
        x = tf(frames).unsqueeze(0).to(device)                            # (1, 3, 16, 112, 112)
        feats.append(model(x).squeeze(0).cpu().numpy())                   # (512,)

feats = np.stack(feats)
np.save("features_r21d.npy", feats)
print(feats.shape)   # (272, 512)

(272, 512)


In [5]:
from pathlib import Path
import cv2

def read_all(path):
    files = sorted(Path(path).glob("*.jpg"), key=lambda f: int(f.stem))
    return np.stack([cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB) for f in files])

def clip_windows(frames, win=16, stride=8):
    T = len(frames)
    if T < win:                                   # короткий клип: растягиваем до 16
        frames = frames[np.linspace(0, T - 1, win).round().astype(int)]
        T = win
    starts = range(0, T - win + 1, stride)
    return torch.stack([tf(torch.from_numpy(frames[s:s + win]).permute(0, 3, 1, 2))
                        for s in starts])        # (окна, 3, 16, 112, 112)

win_feats, win_clip, win_flip = [], [], []
with torch.no_grad():
    for i, path in enumerate(df.path):
        w = clip_windows(read_all(path))                  # (окна, 3, 16, 112, 112)
        for flip in (False, True):
            x = w.flip(-1) if flip else w                 # зеркальное отражение по ширине
            out = torch.cat([model(b.to(device)).cpu() for b in x.split(8)])
            win_feats.append(out.numpy())                 # признак каждого окна отдельно
            win_clip += [i] * len(out)                    # из какого клипа окно
            win_flip += [flip] * len(out)

np.savez("features_r21d_aug.npz",
         feats=np.concatenate(win_feats), clip=np.array(win_clip), flip=np.array(win_flip))

In [6]:
from torchvision.models.video import s3d, S3D_Weights

weights = S3D_Weights.KINETICS400_V1
model = s3d(weights=weights)
model.classifier = nn.Identity()   # вместо 400 классов Kinetics -> вектор 1024
model.eval().to(device)
tf = weights.transforms()          # resize 256, crop 224x224, нормализация Kinetics
print(tf)

Downloading: "https://download.pytorch.org/models/s3d-d76dad2f.pth" to /home/ekaterina/.cache/torch/hub/checkpoints/s3d-d76dad2f.pth


100.0%


VideoClassification(
    crop_size=[224, 224]
    resize_size=[256, 256]
    mean=[0.43216, 0.394666, 0.37645]
    std=[0.22803, 0.22145, 0.216989]
    interpolation=InterpolationMode.BILINEAR
)


In [7]:
feats = []
with torch.no_grad():
    for path in df.path:
        frames = torch.from_numpy(read_frames(path)).permute(0, 3, 1, 2)  # (16, 3, H, W)
        x = tf(frames).unsqueeze(0).to(device)                            # (1, 3, 16, 224, 224)
        feats.append(model(x).squeeze(0).cpu().numpy())                   # (1024,)

feats = np.stack(feats)
np.save("features_s3d.npy", feats)
print(feats.shape)   # (272, 1024)

(272, 1024)
